# Hypothesis Testing – Vanguard A/B Test

This notebook evaluates whether the new digital design performs differently from the traditional design based on selected KPIs.

The analysis uses the pre-computed KPI tables created in the updated project structure:

- `clients_kpi_table.csv`
- `visits_kpi_table.csv`
- `events_kpi_table.csv`

For each KPI, we follow the same structure:

1. Define the KPI and business question
2. Choose the appropriate statistical test
3. Formulate null and alternative hypotheses
4. Run the test
5. Interpret the result

In [1]:
# Import required libraries

import pandas as pd
import numpy as np

from scipy.stats import ttest_ind, chi2_contingency

## Load KPI Tables

We load the pre-computed KPI tables created during the data preparation step.

Each table represents a different analysis level:

- Client level: one row per client
- Visit level: one row per visit
- Event level: one row per interaction/event

In [2]:
# Load pre-computed KPI tables

df_clients = pd.read_csv("../data/clean/client_kpi_table.csv")
df_visits = pd.read_csv("../data/clean/visits_kpi_table.csv")
df_events = pd.read_csv("../data/clean/events_kpi_table.csv")

In [3]:
df_clients.columns

Index(['client_id', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts',
       'bal', 'calls_6_mnth', 'logons_6_mnth', 'Variation',
       'session_per_client', 'max_step_client', 'n_backward_steps',
       'completion_rate', 'avg_steps_per_session', 'avg_error_rate',
       'avg_time_to_completion'],
      dtype='str')

In [4]:
df_visits.columns

Index(['client_id', 'visit_id', 'visit_start', 'visit_end', 'max_step_reached',
       'n_steps', 'n_backward_steps', 'reached_confirm', 'error_rate',
       'visit_duration_sec'],
      dtype='str')

In [5]:
df_events.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time',
       'step_num', 'previous_step_num', 'step_direction', 'time_on_each_step'],
      dtype='str')

### KPI Overview

| KPI                     | Level  | Column                  | Type        | Test        | Definition |
|------------------------|--------|--------------------------|-------------|-------------|------------|
| Completion Rate        | Visit  | reached_confirm          | Binary      | Chi-square  | % of visits reaching the confirm step |
| Time to Completion     | Visit  | visit_duration_sec       | Continuous  | T-test      | Time (in seconds) to complete a visit (only successful visits) |
| Time Spent per Step    | Event  | time_on_each_step        | Continuous  | T-test      | Average time spent on each process step |
| Error Rate             | Visit  | error_rate > 0           | Binary      | Chi-square  | % of visits with at least one error |
| Step Conversion Rate   | Event  | step transitions         | Binary      | Chi-square  | Probability of moving from one step to the next |
| Sessions per Client    | Client | session_per_client       | Continuous  | T-test      | Number of sessions per client |
| Avg Steps per Session  | Client | avg_steps_per_session    | Continuous  | T-test      | Average number of steps per session |

## KPI 1: Completion Rate

### Business Question
Does the new design increase the probability that a user reaches the confirm step?

### Metric
Completion Rate = % of visits where `reached_confirm = 1`

### Statistical Test
Chi-square test of independence

### Hypotheses

- p_test = probability of completion in the test group  
- p_control = probability of completion in the control group  

- **H0 (Null Hypothesis):**  
p_test = p_control  

- **H1 (Alternative Hypothesis):**  
p_test ≠ p_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

In [7]:
# Merge Variation into visits table

df_visits = df_visits.merge(
    df_clients[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

In [9]:
df_visits["Variation"].isna().sum()

np.int64(89687)

In [10]:
# Keep only visits from clients who are part of the experiment
df_visits_ab_test = df_visits[df_visits["Variation"].notna()].copy()

# Check remaining rows by group
df_visits_ab_test["Variation"].value_counts()

Variation
Test       37190
Control    32235
Name: count, dtype: int64

In [11]:
# Create contingency table for Completion Rate

contingency_table = pd.crosstab(
    df_visits_ab_test["Variation"],
    df_visits_ab_test["reached_confirm"]
)

contingency_table

reached_confirm,False,True
Variation,,
Control,16154,16081
Test,15405,21785


In [13]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi2 statistic:", chi2)
print("p-value:", p_value)

Chi2 statistic: 525.6875674507415
p-value: 2.449904079356561e-116


### Interpretation — Completion Rate (Chi-Square Test)

The p-value is extremely small (p < 0.05).

→ We reject the null hypothesis.

There is a statistically significant difference in completion rates between the Test and Control groups.

In [14]:
# Completion rates per group

completion_rates = contingency_table.div(contingency_table.sum(axis=1), axis=0)

completion_rates

reached_confirm,False,True
Variation,,
Control,0.501132,0.498868
Test,0.414224,0.585776


### Business Interpretation

The completion rate is higher in the Test group (~58.6%) compared to the Control group (~49.9%).

→ This indicates that the new design leads to a substantial improvement in completion.

Combined with the statistical significance, this suggests that the observed difference is not due to random variation, but reflects a real positive effect of the new design.

In [15]:
# Extract time to completion per group

control_times = df_visits_ab_test[df_visits_ab_test["Variation"] == "Control"]["visit_duration_sec"]
test_times = df_visits_ab_test[df_visits_ab_test["Variation"] == "Test"]["visit_duration_sec"]

print("Control mean:", control_times.mean())
print("Test mean:", test_times.mean())

Control mean: 280.48317046688385
Test mean: 315.5188760419468


## KPI 2: Time to Completion

### Hypothesis

Let:
- μ_test = average time to completion in the Test group
- μ_control = average time to completion in the Control group

**H0 (Null Hypothesis):**  
μ_test = μ_control  

**H1 (Alternative Hypothesis):**  
μ_test ≠ μ_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

In [16]:
from scipy.stats import ttest_ind

t_stat_time, p_value_time = ttest_ind(test_times, control_times)

print("t-statistic:", t_stat_time)
print("p-value:", p_value_time)

t-statistic: 8.190778265325317
p-value: 2.639072027016892e-16


### Interpretation – Time to Completion (T-Test)

The p-value is extremely small (p < 0.05).

→ We reject the null hypothesis.

There is a statistically significant difference in time to completion between the Test and Control groups.

### Business Interpretation

The average time to completion is higher in the Test group (~315.5 seconds) compared to the Control group (~280.5 seconds).

→ This indicates that users take longer to complete the process in the new design.

Combined with the earlier result (higher completion rate), this suggests a trade-off:

Users are more likely to complete the process in the Test group, but it takes them more time to do so.

This may indicate that the new design provides more guidance or requires more steps, leading to higher completion but longer duration.

## KPI 3: Time Spent on Each Step

### Business Question
Does the new design change how much time users spend on each step?

### Metric
Average time spent per step (`time_on_each_step`) by process step.

### Statistical Test
Two-sample T-test (per step)

### Hypothesis – Time Spent on Step 1

We test whether the average time spent on **step_1** differs between the Test and Control groups.

Let:
- μ_test = average time on step_1 in the Test group  
- μ_control = average time on step_1 in the Control group  

**H0 (Null Hypothesis):**  
μ_test = μ_control  

**H1 (Alternative Hypothesis):**  
μ_test ≠ μ_control  

This is a two-sided test.

We use a two-sample T-test because `time_on_each_step` is a continuous variable and we compare two independent groups.

In [50]:
from scipy.stats import ttest_ind

step1 = df_events_ab_test[df_events_ab_test["process_step"] == "step_1"]

control = step1[step1["Variation"] == "Control"]["time_on_each_step"]
test = step1[step1["Variation"] == "Test"]["time_on_each_step"]

t_stat_1, p_val_1 = ttest_ind(test.dropna(), control.dropna())

print("Step 1 - t-stat:", t_stat_1)
print("Step 1 - p-value:", p_val_1)

Step 1 - t-stat: -6.81820172950013
Step 1 - p-value: 9.295583879256728e-12


### Interpretation

The p-value is far below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the average time spent on step 1 between the Test and Control groups.

Since the t-statistic is negative, users in the Test group spend **less time** on step 1 compared to the Control group.

This suggests that the new design allows users to move through step 1 more efficiently.

In [51]:
step2 = df_events_ab_test[df_events_ab_test["process_step"] == "step_2"]

control = step2[step2["Variation"] == "Control"]["time_on_each_step"]
test = step2[step2["Variation"] == "Test"]["time_on_each_step"]

t_stat_2, p_val_2 = ttest_ind(test.dropna(), control.dropna())

print("Step 2 – t-stat:", t_stat_2)
print("Step 2 – p-value:", p_val_2)

Step 2 – t-stat: 13.026962180397694
Step 2 – p-value: 9.775094850940711e-39


### Interpretation

The p-value is far below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the average time spent on step 2 between the Test and Control groups.

Since the t-statistic is positive, users in the Test group spend **more time** on step 2 compared to the Control group.

This suggests that step 2 in the new design may introduce additional friction or require more effort from users.

In [52]:
step3 = df_events_ab_test[df_events_ab_test["process_step"] == "step_3"]

control = step3[step3["Variation"] == "Control"]["time_on_each_step"]
test = step3[step3["Variation"] == "Test"]["time_on_each_step"]

t_stat_3, p_val_3 = ttest_ind(test.dropna(), control.dropna())

print("Step 3 – t-stat:", t_stat_3)
print("Step 3 – p-value:", p_val_3)

Step 3 – t-stat: 3.6867601857334584
Step 3 – p-value: 0.00022737606203737768


### Interpretation

The p-value is below the significance level (α = 0.05), therefore we reject the null hypothesis.

There is a statistically significant difference in the average time spent on step 3 between the Test and Control groups.

Since the t-statistic is positive, users in the Test group spend **more time** on step 3 compared to the Control group.

This suggests that the new design may introduce additional friction or complexity in step 3.

In [53]:
confirm = df_events_ab_test[df_events_ab_test["process_step"] == "confirm"]

control = confirm[confirm["Variation"] == "Control"]["time_on_each_step"]
test = confirm[confirm["Variation"] == "Test"]["time_on_each_step"]

t_stat_confirm, p_val_confirm = ttest_ind(test.dropna(), control.dropna())

print("Confirm – t-stat:", t_stat_confirm)
print("Confirm – p-value:", p_val_confirm)

Confirm – t-stat: 0.07440334601074908
Confirm – p-value: 0.9406898272960917


### Interpretation

The p-value is above the significance level (α = 0.05), therefore we fail to reject the null hypothesis.

There is no statistically significant difference in the average time spent on the confirm step between the Test and Control groups.

This suggests that the new design does not impact the time users need to complete the final confirmation step.

## KPI 4: Error Rate

### Hypothesis

We test whether the probability of encountering an error differs between the Test and Control groups.

For this analysis, an error is defined as a visit where `error_rate > 0`.

Let:
- p_test = probability of an error in the Test group
- p_control = probability of an error in the Control group

**H0 (Null Hypothesis):**  
p_test = p_control  

**H1 (Alternative Hypothesis):**  
p_test ≠ p_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

In [48]:
# Create contingency table for Error Rate

error_table = pd.crosstab(
    df_visits_ab_test["Variation"],
    df_visits_ab_test["error_rate"] > 0
)

error_table

error_rate,False,True
Variation,,
Control,25539,6696
Test,27078,10112


In [18]:
# Run Chi-Square test for Error Rate
# The contingency table compares Variation with error occurrence (error vs no error)

chi2_error, p_value_error, dof_error, expected_error = chi2_contingency(error_table)

print("Chi2 statistic:", chi2_error)
print("p-value:", p_value_error)

Chi2 statistic: 387.2469792946142
p-value: 3.2901718232687e-86


In [19]:
# Error rates per group
error_rates = error_table.div(error_table.sum(axis=1), axis=0)

error_rates

error_rate,False,True
Variation,,
Control,0.792275,0.207725
Test,0.728099,0.271901


### Interpretation – Error Rate (Chi-Square Test)

The p-value is extremely small (p < 0.05).

→ We reject the null hypothesis.

There is a statistically significant difference in error rates between the Test and Control groups.

The Test group shows a higher error rate (~27.2%) compared to the Control group (~20.8%).

This suggests that users in the new design encounter more difficulties or make more mistakes during the process.

### Business Interpretation

While the Test group has a higher completion rate, it also has a higher error rate.

This suggests a trade-off:
Users are more likely to complete the process, but they experience more friction along the way.

The new design may provide more guidance or require more steps, which increases completion but also introduces more opportunities for errors.

## KPI 5: Step Conversion Rate

### Hypothesis

We analyze whether the probability of progressing from one process step to the next differs between the Test and Control groups, based on event-level data.

Let:
- p_test = probability of progressing to the next step in the Test group
- p_control = probability of progressing to the next step in the Control group

**H0 (Null Hypothesis):**  
p_test = p_control  

**H1 (Alternative Hypothesis):**  
p_test ≠ p_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

Note: We perform a separate Chi-Square test for each step transition using event-level data.

### Prepare Event-Level Dataset

We filter the events dataset to include only users participating in the experiment and merge the group assignment (Test vs Control).

In [30]:
# Keep only events from clients who are part of the experiment
df_events_ab_test = df_events[df_events["client_id"].isin(df_clients["client_id"])].copy()

# Merge Variation into event-level table
df_events_ab_test = df_events_ab_test.merge(
    df_clients[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

# Check group distribution
df_events_ab_test["Variation"].value_counts()

Variation
Test       177787
Control    143408
Name: count, dtype: int64

In [31]:
df_events_ab_test["process_step"].value_counts()

process_step
start      104041
step_1      68410
step_2      56855
step_3      48675
confirm     43214
Name: count, dtype: int64

In [32]:
# Step conversion: start -> step_1

# Users who reached 'start'
start_users = df_events_ab_test[df_events_ab_test["process_step"] == "start"][["client_id", "Variation"]].drop_duplicates()

# Users who reached 'step_1'
step1_users = df_events_ab_test[df_events_ab_test["process_step"] == "step_1"][["client_id"]].drop_duplicates()

# Mark who progressed
start_users["reached_step1"] = start_users["client_id"].isin(step1_users["client_id"])

# Contingency table
step1_table = pd.crosstab(start_users["Variation"], start_users["reached_step1"])

step1_table

reached_step1,False,True
Variation,,
Control,3315,20076
Test,2478,24194


### Step Conversion: Start → Step 1 (Chi-Square Test)

We test whether the probability of progressing from **'start' to 'step_1'** differs between the Control and Test groups.

- **H0 (Null Hypothesis):** Conversion rates are the same in both groups  
- **H1 (Alternative Hypothesis):** Conversion rates differ between groups  

This is a **Chi-Square test of independence**, as we are comparing categorical outcomes (progressed vs not progressed) across groups.

In [33]:
from scipy.stats import chi2_contingency

chi2_step1, p_value_step1, dof_step1, expected_step1 = chi2_contingency(step1_table)

print("Chi2 statistic:", chi2_step1)
print("p-value:", p_value_step1)

Chi2 statistic: 289.73184502312085
p-value: 5.688332164665474e-65


### Interpretation

The p-value is extremely small (p < 0.05), so we reject the null hypothesis.

There is a statistically significant difference in conversion rates between the Test and Control groups for the transition from 'start' to 'step_1'.

The Test group has a higher conversion rate, meaning users are more likely to proceed to the next step under the new design.

This suggests that the redesign improves early-stage progression in the funnel.

Given the very large Chi² statistic and extremely small p-value, the effect is highly unlikely to be due to random variation.

### Data Preparation for Step Conversion Analysis

We use **event-level data** to analyze user progression between steps.

However, since each user can appear multiple times in the dataset (one row per event), using the raw data directly would lead to **double counting users** and biased conversion rates.

To avoid this, we:
- Select only the relevant step (e.g., 'step_1' or 'step_2')
- Keep only the necessary columns (e.g., `client_id`, `Variation`)
- Apply `.drop_duplicates()` to ensure **each user is counted only once per step**

This allows us to correctly measure whether a user progressed from one step to the next.

In [34]:
# Step conversion: step_1 → step_2

# Users who reached 'step_1' (unique users)
step1_users = df_events_ab_test[df_events_ab_test["process_step"] == "step_1"][["client_id", "Variation"]].drop_duplicates()

# Users who reached 'step_2' (unique users)
step2_users = df_events_ab_test[df_events_ab_test["process_step"] == "step_2"][["client_id"]].drop_duplicates()

# Mark who progressed
step1_users["reached_step2"] = step1_users["client_id"].isin(step2_users["client_id"])

# Contingency table
step2_table = pd.crosstab(step1_users["Variation"], step1_users["reached_step2"])

step2_table

reached_step2,False,True
Variation,,
Control,1519,18627
Test,2023,22237


### Step Conversion: Step 1 → Step 2 (Chi-Square Test)

We test whether the probability of progressing from **'step_1' to 'step_2'** differs between the Control and Test groups.

- **H0 (Null Hypothesis):** Conversion rates are the same in both groups  
- **H1 (Alternative Hypothesis):** Conversion rates differ between groups  

This is a **Chi-Square test of independence**, as we compare categorical outcomes (progressed vs not progressed) across groups.

In [35]:
from scipy.stats import chi2_contingency

chi2_step2, p_value_step2, dof_step2, expected_step2 = chi2_contingency(step2_table)

print("Chi2 statistic:", chi2_step2)
print("p-value:", p_value_step2)

Chi2 statistic: 9.460884978856344
p-value: 0.002098997793635178


### Interpretation

The p-value is below 0.05, so we reject the null hypothesis.

There is a statistically significant difference in conversion rates between the Test and Control groups for the transition from 'step_1' to 'step_2'.

However, the Chi² statistic is relatively small compared to the previous step, suggesting that the effect size is more modest.

This indicates that while the redesign still influences user progression at this stage, the impact is weaker than in earlier steps of the funnel.

Overall, the redesign continues to improve progression, but the effect becomes less pronounced deeper in the process.

In [36]:
# Step conversion: step_2 → step_3

# Users who reached 'step_2' (unique users)
step2_users = df_events_ab_test[df_events_ab_test["process_step"] == "step_2"][["client_id", "Variation"]].drop_duplicates()

# Users who reached 'step_3' (unique users)
step3_users = df_events_ab_test[df_events_ab_test["process_step"] == "step_3"][["client_id"]].drop_duplicates()

# Mark who progressed
step2_users["reached_step3"] = step2_users["client_id"].isin(step3_users["client_id"])

# Contingency table
step3_table = pd.crosstab(step2_users["Variation"], step2_users["reached_step3"])

step3_table

reached_step3,False,True
Variation,,
Control,1294,17350
Test,1423,20829


### Step Conversion: Step 2 → Step 3 (Chi-Square Test)

We test whether the probability of progressing from **'step_2' to 'step_3'** differs between the Control and Test groups.

- **H0 (Null Hypothesis):** Conversion rates are the same in both groups  
- **H1 (Alternative Hypothesis):** Conversion rates differ between groups  

This is a **Chi-Square test of independence**, as we compare categorical outcomes (progressed vs not progressed) across groups.

In [37]:
from scipy.stats import chi2_contingency

chi2_step3, p_value_step3, dof_step3, expected_step3 = chi2_contingency(step3_table)

print("Chi2 statistic:", chi2_step3)
print("p-value:", p_value_step3)

Chi2 statistic: 4.781953333642653
p-value: 0.028759480533241844


### Interpretation

The p-value is below 0.05, so we reject the null hypothesis.

There is a statistically significant difference in conversion rates between the Test and Control groups for the transition from 'step_2' to 'step_3'.

However, the Chi² statistic is relatively small, indicating that the effect size is weak.

This suggests that while the redesign still has an impact on user progression at this stage, the influence is minimal compared to earlier steps.

Overall, the effect of the redesign appears to diminish as users move further along the funnel.

This pattern suggests that the redesign primarily reduces friction in the early stages, but has limited impact on deeper steps in the user journey.

In [38]:
# Step conversion: step_3 → confirm

# Users who reached 'step_3' (unique users)
step3_users = df_events_ab_test[
    df_events_ab_test["process_step"] == "step_3"
][["client_id", "Variation"]].drop_duplicates()

# Users who reached 'confirm' (unique users)
confirm_users = df_events_ab_test[
    df_events_ab_test["process_step"] == "confirm"
][["client_id"]].drop_duplicates()

# Mark who progressed to final step
step3_users["reached_confirm"] = step3_users["client_id"].isin(confirm_users["client_id"])

# Contingency table
confirm_table = pd.crosstab(step3_users["Variation"], step3_users["reached_confirm"])

confirm_table

reached_confirm,False,True
Variation,,
Control,2093,15323
Test,2436,18440


### Step Conversion: Step 3 → Confirm (Chi-Square Test)

We test whether the probability of progressing from **'step_3' to 'confirm'** differs between the Control and Test groups.

- **H0 (Null Hypothesis):** Conversion rates are the same in both groups  
- **H1 (Alternative Hypothesis):** Conversion rates differ between groups  

This is a **Chi-Square test of independence**, as we compare categorical outcomes (completed vs not completed) across groups.

In [39]:
from scipy.stats import chi2_contingency

chi2_confirm, p_value_confirm, dof_confirm, expected_confirm = chi2_contingency(confirm_table)

print("Chi2 statistic:", chi2_confirm)
print("p-value:", p_value_confirm)

Chi2 statistic: 1.0743760523799262
p-value: 0.2999590870475827


### Interpretation

The p-value is greater than 0.05, so we fail to reject the null hypothesis.

There is no statistically significant difference in conversion rates between the Test and Control groups for the transition from 'step_3' to 'confirm'.

This suggests that the redesign does not have a meaningful impact on final completion once users reach the last step.

In other words, users who make it to the final stage are equally likely to complete the process in both versions.

### Overall Funnel Insight

The redesign significantly improves user progression in the early stages of the funnel, particularly from 'start' to 'step_1'.

However, the effect diminishes in later steps and disappears entirely at the final stage.

This suggests that the redesign is effective at reducing initial friction and helping users get started, but does not meaningfully impact final completion once users reach the last step.

This pattern indicates that the main opportunity for improvement lies in the early stages of the user journey, rather than in optimizing the final confirmation step.

## KPI 6: Sessions per Client

### Hypothesis

We test whether the average number of sessions per client differs between the Test and Control groups.

Let:
- μ_test = average number of sessions per client in the Test group  
- μ_control = average number of sessions per client in the Control group  

**H0 (Null Hypothesis):**  
μ_test = μ_control  

**H1 (Alternative Hypothesis):**  
μ_test ≠ μ_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

We use a two-sample T-test because sessions per client is a numerical variable and we compare the means of two independent groups.

In [41]:
from scipy.stats import ttest_ind

# Extract sessions per client for each group
control_sessions = df_clients[df_clients["Variation"] == "Control"]["session_per_client"]
test_sessions = df_clients[df_clients["Variation"] == "Test"]["session_per_client"]

# Run independent t-test
t_stat_sessions, p_value_sessions = ttest_ind(test_sessions, control_sessions)

print("t-statistic:", t_stat_sessions)
print("p-value:", p_value_sessions)

t-statistic: 1.255050535340125
p-value: 0.20946622483884744


### Interpretation – Sessions per Client

The p-value (≈ 0.21) is greater than the significance level of 0.05.

→ We fail to reject the null hypothesis.

This means that there is no statistically significant difference in the average number of sessions per client between the Test and Control groups.

### Business Interpretation

The new design does not significantly affect how many sessions users need to complete the process.

This suggests that:

- Users are not required to return more often  
- The increase in completion rate is not driven by repeated attempts  

Overall, the change in design does not negatively impact user efficiency in terms of session frequency.

## KPI 7: Average Steps per Session

### Hypothesis

We test whether the average number of steps per session differs between the Test and Control groups.

Let:
- μ_test = average number of steps per session in the Test group  
- μ_control = average number of steps per session in the Control group  

**H0 (Null Hypothesis):**  
μ_test = μ_control  

**H1 (Alternative Hypothesis):**  
μ_test ≠ μ_control  

This is a two-sided test, as we are interested in detecting any difference between the groups.

We use a two-sample T-test because average steps per session is a numerical variable and we compare the means of two independent groups.

In [42]:
from scipy.stats import ttest_ind

# Extract average steps per session for each group
control_steps = df_clients[df_clients["Variation"] == "Control"]["avg_steps_per_session"]
test_steps = df_clients[df_clients["Variation"] == "Test"]["avg_steps_per_session"]

# Run independent t-test
t_stat_steps, p_value_steps = ttest_ind(test_steps, control_steps)

print("t-statistic:", t_stat_steps)
print("p-value:", p_value_steps)

t-statistic: 16.415652802198654
p-value: 2.1202796357861935e-60


### Interpretation – Average Steps per Session

The p-value is extremely small (p < 0.001), which is far below the significance level of 0.05.

→ We reject the null hypothesis.

This means that there is a statistically significant difference in the average number of steps per session between the Test and Control groups.

### Business Interpretation

The Test group requires significantly more steps per session.

This suggests that:

- The new design introduces additional interactions  
- The process is more complex or less efficient  

This aligns with earlier findings:

- Higher completion rate  
- Longer time to completion  
- More errors  

Together, this indicates a clear trade-off:

The new design improves completion but increases friction along the user journey.

## Final Hypothesis Testing Insights

Across all KPIs, we observe a consistent pattern when comparing the Test and Control groups.

### Key Results

- **Completion Rate**  
  Significantly higher in the Test group  
  → Users are more likely to complete the process

- **Time to Completion**  
  Significantly higher in the Test group  
  → Users take longer to complete the process

- **Time Spent per Step**  
  Mixed results across steps:  
  - Step 1: faster in Test  
  - Steps 2 & 3: slower in Test  
  - Confirm: no significant difference  
  → Initial step is more efficient, but mid-process friction increases

- **Error Rate**  
  Significantly higher in the Test group  
  → Users encounter more errors

- **Sessions per Client**  
  No significant difference  
  → Users do not require more sessions to complete the process

- **Avg Steps per Session**  
  Significantly higher in the Test group  
  → Users go through more steps per session


### Overall Interpretation

The new design improves completion rates, but introduces additional friction throughout the user journey.

This is reflected in:
- longer completion times  
- increased time spent in key steps  
- higher error rates  
- more steps required per session  

However:
- users do not need more sessions to complete the process  


### Business Insight

The new design appears to provide more guidance or structure, which helps users reach completion.

At the same time, it increases complexity and interaction cost.

→ This indicates a clear trade-off between:
- **effectiveness (higher completion)**  
- **efficiency (time, effort, errors)**  


### Recommendation Direction

- Investigate which specific steps introduce friction (especially steps 2 and 3)
- Identify opportunities to simplify the process without reducing completion
- Focus on reducing errors and unnecessary steps in the Test design

Further analysis (e.g., step-level breakdowns and user segmentation) is recommended to isolate where improvements can be made.